Povoamento de Dados- Dimensão Treino

Importação de Pacotes

In [1]:
import pandas as pd
import random

Leitura das Fontes necessárias

In [2]:
atividades = pd.read_csv('../Fontes/atividades_ginasio_fonte.csv', encoding='latin1')
instrutores = pd.read_csv('../Fontes/instrutores_ginasio_fonte.csv', encoding='latin1')
print(f"Registos de atividades lidos: {atividades.shape[0]}")
print(f"Registos de instrutores lidos: {instrutores.shape[0]}")

Registos de atividades lidos: 30
Registos de instrutores lidos: 10


Filtração das Atividades por apenas Circuitos de Treino

In [3]:
treinos = atividades[atividades['tipo_atividade'].str.strip().str.lower() == 'circuito de treino'].copy()
print(f"Treinos identificados: {treinos.shape[0]}")

Treinos identificados: 20


Verificação da Integridade dos Dados

In [4]:
assert treinos['id_atividade'].is_unique, "IDs de treino não são únicos!"
assert treinos[['nome_atividade', 'duracao_minutos', 'calorias_medias_queimadas', 'objetivo_treino']].isnull().sum().sum() == 0, "Existem valores críticos em falta!" 

Processo de Correlação das especialidades dos Instrutores com os Treinos Adequados

In [5]:
instrutores = instrutores.rename(columns={
    'nome': 'instrutor_nome',
    'area_especializacao': 'instrutor_especializacao'
})

In [7]:
# 1. Mapeamento de especialização para keywords
mapa_especializacao = {
    'Musculação': ['musculação', 'bíceps', 'costas', 'peito', 'tríceps', 'ombros', 'pernas'],
    'Yoga': ['yoga'],
    'Pilates': ['pilates'],
    'Cardio': ['cardio', 'hiit', 'corrida', 'spinning'],
    'Crossfit': ['crossfit'],
    'Boxe': ['boxe'],
    'Alongamentos': ['alongamento', 'alongamentos'],
    'Funcional': ['funcional'],
}

def instrutores_possiveis(nome_atividade):
    candidatos = []
    for _, row in instrutores.iterrows():
        area = row['instrutor_especializacao']
        if area in mapa_especializacao:
            if any(kw in nome_atividade.lower() for kw in mapa_especializacao[area]):
                candidatos.append(row['instrutor_nome'])
    return candidatos

treinos['instrutor_nome'] = None
instrutores_usados = {nome: 0 for nome in instrutores['instrutor_nome']}

# 2. Primeiro ciclo: atribuir treinos compatíveis
for _, row in instrutores.iterrows():
    area = row['instrutor_especializacao']
    nome = row['instrutor_nome']
    treinos_disp = treinos[
        treinos['instrutor_nome'].isna() &
        treinos['nome_atividade'].str.lower().apply(lambda x: any(kw in x for kw in mapa_especializacao.get(area, [])))
    ]
    if not treinos_disp.empty:
        i = treinos_disp.sample(1).index[0]
        treinos.at[i, 'instrutor_nome'] = nome
        instrutores_usados[nome] += 1

# 3. Garante que todos os instrutores têm pelo menos um treino
instrutores_sem_treino = [nome for nome, count in instrutores_usados.items() if count == 0]
treinos_disponiveis = treinos[treinos['instrutor_nome'].isna()]

for instrutor_nome in instrutores_sem_treino:
    if not treinos_disponiveis.empty:
        idx = treinos_disponiveis.sample(1).index[0]
        treinos.at[idx, 'instrutor_nome'] = instrutor_nome
        instrutores_usados[instrutor_nome] += 1
        treinos_disponiveis = treinos[treinos['instrutor_nome'].isna()]

# 4. Atribuir os restantes treinos de forma equilibrada (máx 3 por instrutor)
random.seed(42)
for idx, row in treinos[treinos['instrutor_nome'].isna()].iterrows():
    candidatos = instrutores_possiveis(row['nome_atividade'])
    candidatos = [c for c in candidatos if instrutores_usados[c] < 3]
    if candidatos:
        escolhido = random.choice(candidatos)
        treinos.at[idx, 'instrutor_nome'] = escolhido
        instrutores_usados[escolhido] += 1
    else:
        candidatos = [c for c in instrutores['instrutor_nome'] if instrutores_usados[c] < 3]
        if candidatos:
            escolhido = random.choice(candidatos)
            treinos.at[idx, 'instrutor_nome'] = escolhido
            instrutores_usados[escolhido] += 1

In [8]:
# 5. Construir a dimensão treino final
dim_treino = treinos[[
    'id_atividade', 'nome_atividade', 'duracao_minutos', 'calorias_medias_queimadas',
    'nivel_dificuldade', 'objetivo_treino', 'instrutor_nome'
]].copy()

dim_treino = dim_treino.rename(columns={
    'id_atividade': 'treino_id_atividade',
    'nome_atividade': 'treino_nome',
    'duracao_minutos': 'treino_duracao_min',
    'calorias_medias_queimadas': 'treino_calorias',
    'nivel_dificuldade': 'treino_dificuldade',
    'objetivo_treino': 'treino_objetivo',
})
dim_treino.insert(0, 'treino_sk', range(1, 1 + len(dim_treino)))

In [10]:
dim_treino.to_csv("../Dados Finais/dim_treino.csv", index=False, encoding="utf-8-sig")
print("Dimensão Treino pronta para carga!")
dim_treino.head()

Dimensão Treino pronta para carga!


,treino_sk,treino_id_atividade,treino_nome,treino_duracao_min,treino_calorias,treino_dificuldade,treino_objetivo,instrutor_nome
10,1,11,Circuito de Bíceps,26,523,Baixo,Ganho de massa,Enzo Pacheco
11,2,12,Circuito de Costas,45,595,Baixo,Resistência,Enzo Pacheco
12,3,13,Circuito de Pernas,47,616,Alto,Relaxamento,Enzo Pacheco
13,4,14,Circuito de Peito,34,284,Médio,Resistência,Leandro Amaral
14,5,15,Circuito de Ombros,37,637,Médio,Condicionamento físico,Marcos Castro Carneiro
